# Code Indexing and Search

This notebook demonstrates a code indexing agent that uses a three-index architecture
(code, descriptions, breadcrumbs) to understand and navigate codebases. We first index
a directory (a user-initiated setup step), then let the agent search and navigate the
pre-built index across multiple tasks.

**Prerequisites**: An embedding provider must be running (e.g. `ollama serve`) and an
LLM API key must be configured, since indexing generates both embeddings and LLM-powered
descriptions for each code symbol.

In [ ]:
from pathlib import Path

from agentic_patterns.agents.code_index import create_agent
from agentic_patterns.core.agents import run_agent
from agentic_patterns.core.config.config import MAIN_PROJECT_DIR
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance
from agentic_patterns.core.rag.chunker_code import ChunkerCode
from agentic_patterns.toolkits.code_index.code_index import CodeIndex
from agentic_patterns.toolkits.code_index.models import PrintIndexListener
from agentic_patterns.tools.code_index import register_index

## Syntax-aware chunking (the foundation)

Before using the agent, let's see what it works with. `ChunkerCode` uses Tree-sitter
to parse source files and extract functions, classes, and methods as individual chunks.
Each chunk carries symbol metadata (name, type, line range) that the agent's tools use
for navigation.

In [ ]:
sample_code = '''
import os
from pathlib import Path

MAX_RETRIES = 3


def connect(host: str, port: int) -> bool:
    """Establish connection to the server."""
    for attempt in range(MAX_RETRIES):
        try:
            return _try_connect(host, port)
        except OSError:
            continue
    return False


class ConnectionPool:
    def __init__(self, size: int = 10):
        self._pool = []
        self._size = size

    def acquire(self) -> object:
        if self._pool:
            return self._pool.pop()
        return object()

    def release(self, conn: object) -> None:
        if len(self._pool) < self._size:
            self._pool.append(conn)
'''

chunker = ChunkerCode()
provenance = DocumentProvenance(original_file=Path("example.py"), source="example.py")
chunks = chunker.chunk(sample_code, provenance)

for chunk in chunks:
    symbol = chunk.metadata.get("symbol_name", "")
    symbol_type = chunk.metadata.get("symbol_type", "")
    print(f"{chunk.doc_id} [{chunk.level.value}] {symbol_type}:{symbol}")

## Indexing (setup)

Indexing is a user-initiated step, not something the agent decides to do. We create a
`CodeIndex`, call `index()` to populate the three collections (code, descriptions,
breadcrumbs), and register it with a description so the agent can discover it via
semantic search -- the user never needs to mention collection names.

In [ ]:
target_dir = MAIN_PROJECT_DIR / "agentic_patterns" / "core" / "rag"

code_index = CodeIndex(target_dir, "code_demo")
stats = await code_index.index(include_patterns=["*.py"], listener=PrintIndexListener())

register_index(code_index, description="RAG pipeline: chunking, clustering, retrieval, and code-aware parsing")

## The agent

The agent has four tools: `code_list_indexes` (discover relevant collections by semantic
search on their descriptions), `code_search` (semantic search across code, descriptions,
and breadcrumbs), `code_expand` (navigate from a symbol to its parent, siblings, and full
context), and `code_lexical_search` (exact/regex match on source files). It operates on a
pre-built index -- it does not decide when to index.

### Task 1: Overview

Ask the agent to explore the indexed codebase and summarize it. It will use
`code_list_indexes` to find the right collection, then `code_search` to find the main
symbols and their descriptions.

In [ ]:
agent = create_agent()

agent_run, _ = await run_agent(
    agent,
    "Give me an overview of the main classes and their responsibilities "
    "in the RAG pipeline code.",
    verbose=True,
)
if agent_run:
    print(agent_run.result.output)

### Task 2: Intent-level search

Ask a "how does X work" question. The agent should search the descriptions index
(matching against LLM-generated summaries) and then expand results to show
code and structural context.

In [ ]:
agent_run, _ = await run_agent(
    agent,
    "How does the system decide when to use tree-sitter vs. the fallback chunker? "
    "Show me the relevant code.",
    verbose=True,
)
if agent_run:
    print(agent_run.result.output)

### Task 3: Structural navigation

Ask about a specific symbol's relationships. The agent should use `code_search` to find
it, then `code_expand` to navigate to its parent class, sibling methods, and breadcrumb
trail.

In [ ]:
agent_run, _ = await run_agent(
    agent,
    "Find the ChunkerCode class, expand it, and list all its methods with a one-line "
    "description of each. Also show what other chunkers exist in the same module.",
    verbose=True,
)
if agent_run:
    print(agent_run.result.output)

### Task 4: Lexical search + cross-referencing

Ask the agent to find exact usages of a symbol across the codebase. This exercises
`code_lexical_search` (regex on source files) combined with `code_search` to understand
what each usage site does.

In [ ]:
agent_run, _ = await run_agent(
    agent,
    "Where is ChunkerSmart referenced in the code? For each reference, explain "
    "what role it plays there.",
    verbose=True,
)
if agent_run:
    print(agent_run.result.output)